# Gradient descent — companion notebook

Runnable companion for [`gradient-descent.md`](./gradient-descent.md).

1. Minimize a 2D ill-conditioned quadratic with fixed step, backtracking line search, and momentum.
2. Demonstrate divergence when the step is too large.
3. Plot iterate trajectories on the loss contour.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# Ill-conditioned quadratic: f(x) = 0.5 x^T A x
A = np.array([[20.0, 0.0],
              [0.0, 1.0]])
L = float(np.linalg.eigvalsh(A).max())
m = float(np.linalg.eigvalsh(A).min())
kappa = L / m
print(f'L = {L},  m = {m},  condition number kappa = {kappa}')

def f(x):
    return 0.5 * x @ A @ x

def grad(x):
    return A @ x

## 1. Three optimizers

- Fixed step at the safe size $\eta = 1/L$.
- Backtracking (Armijo) line search.
- Heavy-ball momentum with $\beta = 0.9$.

In [ ]:
def gd_fixed(x0, eta, n_iter):
    xs = [x0.copy()]
    x = x0.copy()
    for _ in range(n_iter):
        x = x - eta * grad(x)
        xs.append(x.copy())
    return np.array(xs)

def gd_backtracking(x0, n_iter, eta0=1.0, c=1e-4, rho=0.5):
    xs = [x0.copy()]
    x = x0.copy()
    for _ in range(n_iter):
        g = grad(x)
        eta = eta0
        while f(x - eta * g) > f(x) - c * eta * (g @ g):
            eta *= rho
            if eta < 1e-12:
                break
        x = x - eta * g
        xs.append(x.copy())
    return np.array(xs)

def gd_momentum(x0, eta, beta, n_iter):
    xs = [x0.copy()]
    x = x0.copy()
    v = np.zeros_like(x)
    for _ in range(n_iter):
        v = beta * v - eta * grad(x)
        x = x + v
        xs.append(x.copy())
    return np.array(xs)

x0 = np.array([1.0, 10.0])
n_iter = 60

traj_fixed = gd_fixed(x0, eta=1.0 / L, n_iter=n_iter)
traj_bt    = gd_backtracking(x0, n_iter=n_iter)
traj_mom   = gd_momentum(x0, eta=1.0 / L, beta=0.9, n_iter=n_iter)

for name, traj in [('fixed 1/L', traj_fixed), ('backtracking', traj_bt), ('momentum', traj_mom)]:
    print(f'{name:>14s}: f after {n_iter} iter = {f(traj[-1]):.3e}')

## 2. Divergence with too-large step

Theory: GD on this quadratic diverges for $\eta > 2/L = $ `2/L`.

In [ ]:
traj_div = gd_fixed(x0, eta=2.1 / L, n_iter=30)
fs_div = np.array([f(x) for x in traj_div])
print(f'2/L = {2/L:.4f}; using eta = {2.1/L:.4f}')
print('f values (first 10):', fs_div[:10])
print('f values explode -> diverged')

## 3. Trajectory plot on contours

The narrow valley (high $\kappa$) makes plain fixed GD zig-zag; momentum cuts diagonally; backtracking adapts step size at each iterate.

In [ ]:
xs = np.linspace(-3, 3, 200)
ys = np.linspace(-12, 12, 200)
X, Y = np.meshgrid(xs, ys)
Z = 0.5 * (A[0, 0] * X**2 + A[1, 1] * Y**2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.contour(X, Y, Z, levels=np.logspace(-1, 2.5, 18), colors='lightgray')
for traj, label, color in [
    (traj_fixed, 'fixed 1/L', 'tab:blue'),
    (traj_bt,    'backtracking', 'tab:green'),
    (traj_mom,   'momentum',  'tab:red'),
]:
    ax.plot(traj[:, 0], traj[:, 1], '-o', ms=3, color=color, label=label, alpha=0.8)
ax.plot(0, 0, 'k*', ms=14, label='optimum')
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('Trajectories on contour of f')
ax.legend()

ax = axes[1]
for traj, label, color in [
    (traj_fixed, 'fixed 1/L', 'tab:blue'),
    (traj_bt,    'backtracking', 'tab:green'),
    (traj_mom,   'momentum',  'tab:red'),
]:
    fs = np.array([f(x) for x in traj])
    ax.semilogy(fs, color=color, label=label)
ax.set_xlabel('iteration'); ax.set_ylabel('f(x_k)  (log)')
ax.set_title('Convergence of objective')
ax.legend(); ax.grid(True, which='both', alpha=0.3)

plt.tight_layout(); plt.show()